# RL for Optimal Trade Execution

A PPO agent that makes per-second execution decisions based on the current order book state.
Instead of predicting 60 seconds ahead, the agent reacts to what it sees right now.

In [1]:
import os, sys, subprocess

# Colab setup
if os.path.exists("/content"):
    # Always ensure we're in the right directory
    repo_dir = "/content/Ultramarin"

    # Clone repo if needed (for simulate_walk_the_book.py)
    if not os.path.isdir(os.path.join(repo_dir, "utils")):
        subprocess.run(["git", "clone", "-q", "-b", "potentially-improve-model",
                        "https://github.com/JosephZacharyGawlik/Ultramarin.git"],
                       cwd="/content")

    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)

    # Mount Drive and copy data if needed
    dst = os.path.join(repo_dir, "data")
    if not os.path.isdir(dst) or not any(
        d for d in os.listdir(dst) if os.path.isdir(os.path.join(dst, d))
    ):
        from google.colab import drive
        drive.mount('/content/drive')
        src = "/content/drive/MyDrive/data"
        os.makedirs(dst, exist_ok=True)
        subprocess.run(["cp", "-r", f"{src}/.", dst], capture_output=True, text=True)
        print("Data copied.")
    else:
        print(f"Data already present at {dst}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gymnasium", "stable-baselines3"], capture_output=True)
print("Setup complete.")

Data already present at /content/Ultramarin/data
Setup complete.


In [2]:
import numpy as np
import pandas as pd
import math
import warnings
import random
from pathlib import Path

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

from data.simulate_walk_the_book import simulate_walk_the_book

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

In [3]:
# === Cross-currency training: train on ALL currencies ===
ALL_CURRENCIES = ["BTCUSDT", "ETHUSDT", "LTCUSDT", "SOLUSDT", "ADAUSDT", "DOGEUSDT", "XRPUSDT"]

KNOWN_VOLUMES = {
    "BTCUSDT": 4.0, "ETHUSDT": 55.0, "LTCUSDT": 51.0,
    "SOLUSDT": 315.0, "ADAUSDT": 10436.0, "DOGEUSDT": 60349.0, "XRPUSDT": 13736.0,
}
OPTIMAL_K = {
    "BTCUSDT": 14, "ETHUSDT": 26, "LTCUSDT": 16,
    "SOLUSDT": 17, "ADAUSDT": 7, "DOGEUSDT": 20, "XRPUSDT": 20,
}

root = Path("data")

print("Cross-currency training mode")
print(f"Currencies: {len(ALL_CURRENCIES)}")
for c in ALL_CURRENCIES:
    print(f"  {c}: volume={KNOWN_VOLUMES[c]}, K={OPTIMAL_K[c]}")

Cross-currency training mode
Currencies: 7
  BTCUSDT: volume=4.0, K=14
  ETHUSDT: volume=55.0, K=26
  LTCUSDT: volume=51.0, K=16
  SOLUSDT: volume=315.0, K=17
  ADAUSDT: volume=10436.0, K=7
  DOGEUSDT: volume=60349.0, K=20
  XRPUSDT: volume=13736.0, K=20


## Data Loading

Load Y data (the 60-second execution window), split into train/val using the same
chrono split as mhf-final (sort IDs, 80/20 split), and pre-extract LOB arrays per instrument.

In [4]:
ASK_PRICE_COLS = [f'ask_price_{i}' for i in range(1, 6)]
ASK_VOL_COLS = [f'ask_vol_{i}' for i in range(1, 6)]
BID_PRICE_COLS = [f'bid_price_{i}' for i in range(1, 6)]
BID_VOL_COLS = [f'bid_vol_{i}' for i in range(1, 6)]


def load_currency(data_asset):
    """Load and clean Y data for one currency, return train/val instrument lists."""
    Y_path = root / data_asset / "y_train.parquet"
    Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])

    price_cols = ASK_PRICE_COLS + BID_PRICE_COLS
    vol_cols = ASK_VOL_COLS + BID_VOL_COLS
    Y_raw[price_cols] = Y_raw.groupby("anonymized_id")[price_cols].transform(
        lambda s: s.ffill().bfill()
    )
    Y_raw[vol_cols] = Y_raw[vol_cols].fillna(0.0)

    id_counts = Y_raw.groupby("anonymized_id").size()
    valid_ids = sorted(id_counts[id_counts == 60].index.tolist())
    split_idx = int(len(valid_ids) * 0.8)
    train_ids = valid_ids[:split_idx]
    val_ids = valid_ids[split_idx:]

    volume = KNOWN_VOLUMES[data_asset]
    K = OPTIMAL_K[data_asset]

    def extract(ids):
        instruments = []
        for aid in ids:
            df_inst = Y_raw[Y_raw["anonymized_id"] == aid].sort_values("time_in_hour")
            if len(df_inst) != 60:
                continue
            ask_prices = df_inst[ASK_PRICE_COLS].to_numpy(dtype=np.float64)
            bid_prices = df_inst[BID_PRICE_COLS].to_numpy(dtype=np.float64)
            ask_vols = df_inst[ASK_VOL_COLS].to_numpy(dtype=np.float64)
            bid_vols = df_inst[BID_VOL_COLS].to_numpy(dtype=np.float64)
            if np.isnan(ask_prices[:, 0]).any() or np.isnan(bid_prices[:, 0]).any():
                continue
            close_vals = df_inst["close"].dropna()
            if len(close_vals) == 0:
                continue
            instruments.append({
                "id": aid,
                "currency": data_asset,
                "volume_to_fill": volume,
                "K": K,
                "ask_prices": ask_prices,
                "ask_vols": ask_vols,
                "bid_prices": bid_prices,
                "bid_vols": bid_vols,
                "close": close_vals.iloc[-1],
            })
        return instruments

    train = extract(train_ids)
    val = extract(val_ids)
    return train, val, Y_raw


# Load all currencies
all_train = []
all_val = {}  # per-currency val sets for evaluation
all_Y_raw = {}

for currency in ALL_CURRENCIES:
    train, val, Y_raw = load_currency(currency)
    all_train.extend(train)
    all_val[currency] = val
    all_Y_raw[currency] = Y_raw
    print(f"  {currency}: {len(train)} train, {len(val)} val")

print(f"\nTotal training instruments: {len(all_train)}")
print(f"Total val instruments: {sum(len(v) for v in all_val.values())}")

  BTCUSDT: 564 train, 141 val


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  ETHUSDT: 562 train, 141 val


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  LTCUSDT: 311 train, 78 val


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  DOGEUSDT: 475 train, 119 val


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  XRPUSDT: 521 train, 131 val

Total training instruments: 3321
Total val instruments: 833


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Gymnasium Environment

Each episode = one instrument's last K-second execution window (same window as TWAP-K and Oracle).  
State: 12 features (remaining vol/time, spread, depth, imbalance, momentum, etc.)  
Action: 6 discrete choices — multiples of the TWAP-K rate  

**Key design: agent only trades in the last K seconds.**  
This matches the Oracle's optimization window and TWAP-K's execution window.  
Executing earlier hurts BPS because prices drift from the close price.

**Reward (differential mark-to-market vs TWAP-K):**  
At each step, compare agent's cost-to-complete vs TWAP-K's cost-to-complete.  
Reward = change in advantage. Positive reward = agent is doing better than TWAP-K.

In [5]:
class TradeExecutionEnv(gym.Env):
    """
    Cross-currency RL environment for optimal trade execution.

    Each instrument carries its own volume_to_fill and K. On reset, the
    environment adapts episode length and TWAP rate to the selected instrument.
    Observation features are normalized (fractions/ratios), so the agent sees
    comparable states across currencies.

    Reward: Differential mark-to-market advantage over TWAP-K.
    Action space: 10 discrete actions from 0x to 3x TWAP-K rate.
    """

    REWARD_SCALE = 10000
    MAX_K = 26  # longest episode (ETH)

    def __init__(self, instruments):
        super().__init__()
        self.instruments = instruments

        self.action_space = spaces.Discrete(10)
        self.action_multipliers = np.array([
            0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0
        ])

        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(12,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        if options and "instrument_idx" in options:
            idx = options["instrument_idx"]
        else:
            idx = self.np_random.integers(len(self.instruments))
        self.inst = self.instruments[idx]

        # Per-instrument parameters
        self.volume_to_fill = self.inst["volume_to_fill"]
        self.K = self.inst["K"]
        self.start_second = 60 - self.K
        self.twap_rate = self.volume_to_fill / self.K

        self.t = 0
        self.volume_executed = 0.0
        self.value_executed = 0.0
        self.volume_allocated = 0.0
        self.remaining_carry = 0.0
        self.prev_advantage = 0.0

        self._precompute_twap()

        abs_t = self.start_second
        self.arrival_mid = (
            self.inst["ask_prices"][abs_t, 0] + self.inst["bid_prices"][abs_t, 0]
        ) / 2

        return self._get_obs(), {}

    def _precompute_twap(self):
        self.twap_cost_per_step = np.zeros(self.K)
        self.twap_vol_per_step = np.zeros(self.K)
        carry = 0.0
        for step in range(self.K):
            abs_t = self.start_second + step
            to_fill = self.twap_rate + carry
            carry = 0.0
            filled = 0.0
            cost = 0.0
            remaining = to_fill
            for level in range(5):
                price = self.inst["ask_prices"][abs_t, level]
                avail = self.inst["ask_vols"][abs_t, level]
                if math.isnan(price) or math.isnan(avail):
                    continue
                take = min(remaining, avail)
                cost += take * price
                filled += take
                remaining -= take
                if remaining <= 1e-12:
                    break
            carry = remaining
            self.twap_cost_per_step[step] = cost
            self.twap_vol_per_step[step] = filled
        self.twap_cum_cost = np.cumsum(self.twap_cost_per_step)
        self.twap_cum_vol = np.cumsum(self.twap_vol_per_step)

    def _get_obs(self):
        if self.t >= self.K:
            return np.zeros(12, dtype=np.float32)

        abs_t = self.start_second + self.t
        ask_p = self.inst["ask_prices"][abs_t]
        ask_v = self.inst["ask_vols"][abs_t]
        bid_p = self.inst["bid_prices"][abs_t]
        bid_v = self.inst["bid_vols"][abs_t]
        mid = (ask_p[0] + bid_p[0]) / 2

        remaining_vol = max(0, self.volume_to_fill - self.volume_allocated)
        remaining_vol_frac = remaining_vol / (self.volume_to_fill + 1e-8)
        remaining_time_frac = (self.K - self.t) / self.K
        spread = (ask_p[0] - bid_p[0]) / (mid + 1e-8)
        mid_return = (mid - self.arrival_mid) / (self.arrival_mid + 1e-8)

        vol_imb_1 = (bid_v[0] - ask_v[0]) / (bid_v[0] + ask_v[0] + 1e-8)
        vol_imb_total = (bid_v.sum() - ask_v.sum()) / (bid_v.sum() + ask_v.sum() + 1e-8)
        ask_depth = ask_v.sum() / (self.volume_to_fill + 1e-8)
        bid_depth = bid_v.sum() / (self.volume_to_fill + 1e-8)

        cum_is = 0.0
        if self.volume_executed > 0:
            avg_price = self.value_executed / self.volume_executed
            cum_is = (avg_price - self.arrival_mid) / (self.arrival_mid + 1e-8)

        if abs_t >= 5:
            mid_prev = (self.inst["ask_prices"][abs_t-5, 0] + self.inst["bid_prices"][abs_t-5, 0]) / 2
            momentum = (mid - mid_prev) / (self.arrival_mid + 1e-8)
            prev_spread = (self.inst["ask_prices"][abs_t-5, 0] - self.inst["bid_prices"][abs_t-5, 0]) / (mid_prev + 1e-8)
            spread_change = spread - prev_spread
        else:
            momentum = 0.0
            spread_change = 0.0

        carry_frac = self.remaining_carry / (self.volume_to_fill + 1e-8)

        obs = np.array([
            remaining_vol_frac,
            remaining_time_frac,
            spread,
            mid_return,
            vol_imb_1,
            vol_imb_total,
            ask_depth,
            bid_depth,
            cum_is,
            momentum,
            spread_change,
            carry_frac,
        ], dtype=np.float32)

        obs = np.nan_to_num(obs, nan=0.0, posinf=10.0, neginf=-10.0)
        obs = np.clip(obs, -10.0, 10.0)
        return obs

    def step(self, action):
        position = self.action_multipliers[action] * self.twap_rate

        max_remaining = self.volume_to_fill - self.volume_allocated
        position = min(position, max(0.0, max_remaining))

        if self.t == self.K - 1:
            position = max(0.0, max_remaining)

        self.volume_allocated += position

        abs_t = self.start_second + self.t
        to_fill = position + self.remaining_carry
        self.remaining_carry = 0.0
        step_volume = 0.0
        step_cost = 0.0

        if to_fill > 1e-12:
            remaining = to_fill
            for level in range(5):
                price = self.inst["ask_prices"][abs_t, level]
                avail = self.inst["ask_vols"][abs_t, level]
                if math.isnan(price) or math.isnan(avail):
                    continue
                take = min(remaining, avail)
                step_cost += take * price
                step_volume += take
                remaining -= take
                if remaining <= 1e-12:
                    break
            self.remaining_carry = remaining

        self.volume_executed += step_volume
        self.value_executed += step_cost

        mid_t = (self.inst["ask_prices"][abs_t, 0] + self.inst["bid_prices"][abs_t, 0]) / 2
        if math.isnan(mid_t):
            mid_t = self.arrival_mid

        agent_remaining = self.volume_to_fill - self.volume_executed
        agent_total_cost = self.value_executed + agent_remaining * mid_t

        twap_remaining = self.volume_to_fill - self.twap_cum_vol[self.t]
        twap_total_cost = self.twap_cum_cost[self.t] + twap_remaining * mid_t

        normalizer = self.arrival_mid * self.volume_to_fill
        advantage = (twap_total_cost - agent_total_cost) / normalizer

        reward = (advantage - self.prev_advantage) * self.REWARD_SCALE
        self.prev_advantage = advantage

        reward = max(-100.0, min(100.0, reward))

        self.t += 1
        done = self.t >= self.K
        info = {}

        if done:
            fill_ratio = self.volume_executed / (self.volume_to_fill + 1e-8)
            if fill_ratio < 0.99:
                reward -= (1.0 - fill_ratio) * 100.0

            if self.volume_executed > 0:
                avg_price = self.value_executed / self.volume_executed
                close = self.inst["close"]
                bps = abs(avg_price - close) / close * 10000
                vol_penalty = min(100.0, self.volume_to_fill / self.volume_executed)
                info["bps"] = bps * vol_penalty
                info["fill_ratio"] = fill_ratio
                info["currency"] = self.inst["currency"]
            else:
                info["bps"] = 10000.0
                info["fill_ratio"] = 0.0
                info["currency"] = self.inst["currency"]

        obs = self._get_obs()
        return obs, reward, done, False, info


# === Sanity checks with mixed currencies ===
test_env = TradeExecutionEnv(all_train)

for currency in ["BTCUSDT", "SOLUSDT", "DOGEUSDT"]:
    # Find an instrument of this currency
    idx = next(i for i, inst in enumerate(all_train) if inst["currency"] == currency)
    obs, _ = test_env.reset(options={"instrument_idx": idx})
    total_reward = 0
    K = all_train[idx]["K"]
    for _ in range(K):
        obs, reward, done, _, info = test_env.step(4)  # 1.0x TWAP
        total_reward += reward
        if done:
            break
    print(f"{currency}: K={K}, BPS={info.get('bps', 'N/A'):.4f}, "
          f"Fill={info.get('fill_ratio', 0):.2%}, Reward={total_reward:.4f}")

nan_count = 0
for _ in range(50):
    obs, _ = test_env.reset()
    K = test_env.K
    for _ in range(K):
        if np.isnan(obs).any():
            nan_count += 1
        obs, _, done, _, _ = test_env.step(test_env.action_space.sample())
        if done:
            break
print(f"\nNaN check: {nan_count} NaN observations in 50 episodes (should be 0)")
print(f"Total training instruments: {len(all_train)}")

BTCUSDT: K=14, BPS=1.1639, Fill=92.81%, Reward=-7.1931
SOLUSDT: K=17, BPS=2.3080, Fill=100.00%, Reward=0.0000
DOGEUSDT: K=20, BPS=0.9775, Fill=100.00%, Reward=0.0000

NaN check: 0 NaN observations in 50 episodes (should be 0)
Total training instruments: 3321


## Training

PPO (Proximal Policy Optimization) with a small MLP policy (2 hidden layers, 64 neurons each).
PPO uses trajectory rollouts rather than replay buffers, making it better suited for
environments with small per-step reward signals. RL-Exec (BTC-USD execution) used PPO
for this exact reason.

In [ ]:
from stable_baselines3.common.vec_env import SubprocVecEnv

NUM_ENVS = 4

def make_env(instruments, seed):
    def _init():
        env = TradeExecutionEnv(instruments)
        env.reset(seed=seed)
        return env
    return _init

train_env = SubprocVecEnv([make_env(all_train, SEED + i) for i in range(NUM_ENVS)])

def linear_schedule(progress_remaining: float) -> float:
    return progress_remaining * 3e-4

model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=linear_schedule,
    n_steps=2048 // NUM_ENVS,  # per-env steps (total batch = n_steps * NUM_ENVS = 2048)
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs={"net_arch": [64, 64]},
    verbose=1,
    seed=SEED,
)

TOTAL_TIMESTEPS = 1_000_000
print(f"Training on {len(all_train)} instruments across {len(ALL_CURRENCIES)} currencies")
print(f"Training for {TOTAL_TIMESTEPS} timesteps ({NUM_ENVS} parallel envs)")
print(f"LR schedule: 3e-4 → 0 (linear decay)")
model.learn(total_timesteps=TOTAL_TIMESTEPS)
train_env.close()
print("Training complete.")

## Evaluation

Run the trained agent on validation instruments and compare BPS against TWAP-K.  
Uses `simulate_walk_the_book()` for final BPS — same function as all other strategies.

In [ ]:
def evaluate_currency(model, instruments, volume_to_fill, K_seconds):
    """Evaluate RL agent vs TWAP baselines on one currency's val set."""
    env = TradeExecutionEnv(instruments)  # instruments already have volume/K
    rl_bps_list = []
    twap_k_bps_list = []
    twap_14_bps_list = []
    rl_positions_all = []

    for i in range(len(instruments)):
        inst = instruments[i]
        ask_prices = inst["ask_prices"]
        ask_vols = inst["ask_vols"]
        bid_prices = inst["bid_prices"]
        bid_vols = inst["bid_vols"]
        close_price = inst["close"]

        # --- RL Agent ---
        obs, _ = env.reset(options={"instrument_idx": i})
        rl_positions = np.zeros(60)
        for step in range(K_seconds):
            action, _ = model.predict(obs, deterministic=True)

            position = env.action_multipliers[int(action)] * env.twap_rate
            max_remaining = volume_to_fill - env.volume_allocated
            position = min(position, max(0.0, max_remaining))
            if step == K_seconds - 1:
                position = max(0.0, max_remaining)

            abs_t = env.start_second + step
            rl_positions[abs_t] = position

            obs, _, done, _, _ = env.step(int(action))
            if done:
                break

        rl_positions_all.append(rl_positions)

        def compute_bps(positions):
            vol, avg = simulate_walk_the_book(
                positions, ask_prices, ask_vols, bid_prices, bid_vols
            )
            if vol > 0 and not np.isnan(avg):
                ie = abs(avg - close_price) / close_price * 10000
                vp = min(100.0, volume_to_fill / vol)
                return ie * vp
            return None

        bps = compute_bps(rl_positions)
        if bps is not None:
            rl_bps_list.append(bps)

        twap_k_pos = np.zeros(60)
        twap_k_pos[-K_seconds:] = volume_to_fill / K_seconds
        bps = compute_bps(twap_k_pos)
        if bps is not None:
            twap_k_bps_list.append(bps)

        twap_14_pos = np.zeros(60)
        twap_14_pos[-14:] = volume_to_fill / 14
        bps = compute_bps(twap_14_pos)
        if bps is not None:
            twap_14_bps_list.append(bps)

    return np.array(rl_bps_list), np.array(twap_k_bps_list), np.array(twap_14_bps_list), rl_positions_all


# === Evaluate per currency ===
print(f"\n{'='*70}")
print(f"CROSS-CURRENCY RL RESULTS (trained on all 7 currencies)")
print(f"{'='*70}")
print(f"{'Currency':<12} {'RL BPS':>8} {'TWAP-K':>8} {'Diff':>8} {'Win%':>7} {'N':>5}")
print(f"{'-'*70}")

all_results = {}
for currency in ALL_CURRENCIES:
    val_insts = all_val[currency]
    if len(val_insts) == 0:
        continue
    volume = KNOWN_VOLUMES[currency]
    K = OPTIMAL_K[currency]

    rl_bps, twap_k_bps, twap_14_bps, positions = evaluate_currency(
        model, val_insts, volume, K
    )

    diff = twap_k_bps.mean() - rl_bps.mean()
    wins = (rl_bps < twap_k_bps).sum()
    pct = (rl_bps < twap_k_bps).mean() * 100
    label = "+" if diff > 0 else ""

    print(f"{currency:<12} {rl_bps.mean():>8.4f} {twap_k_bps.mean():>8.4f} "
          f"{label}{diff:>7.4f} {pct:>6.1f}% {len(rl_bps):>5}")

    all_results[currency] = {
        "rl_bps": rl_bps, "twap_k_bps": twap_k_bps,
        "twap_14_bps": twap_14_bps, "positions": positions,
    }

print(f"{'-'*70}")
print(f"\nComparison with per-currency RL agents:")
print(f"{'Currency':<12} {'Cross-RL':>10} {'Single-RL':>10} {'TWAP-K':>8}")
print(f"{'-'*50}")
single_results = {
    "SOLUSDT": 4.0119, "XRPUSDT": 3.1492, "LTCUSDT": 4.2573,
    "BTCUSDT": 1.4437, "ETHUSDT": 2.8400, "DOGEUSDT": 4.9916,
}
for currency in ALL_CURRENCIES:
    if currency in all_results:
        cross = all_results[currency]["rl_bps"].mean()
        single = single_results.get(currency, "—")
        twap = all_results[currency]["twap_k_bps"].mean()
        if isinstance(single, float):
            print(f"{currency:<12} {cross:>10.4f} {single:>10.4f} {twap:>8.4f}")
        else:
            print(f"{currency:<12} {cross:>10.4f} {'—':>10} {twap:>8.4f}")

: 

: 

: 

## Policy Analysis

Examine what the agent learned: average position profile across instruments,
compared to TWAP allocation.

In [ ]:
# Per-currency policy analysis
for currency in ALL_CURRENCIES:
    if currency not in all_results:
        continue
    positions = all_results[currency]["positions"]
    K = OPTIMAL_K[currency]
    volume = KNOWN_VOLUMES[currency]

    positions_array = np.array(positions)
    avg_positions = positions_array.mean(axis=0)

    print(f"\n{currency} (volume={volume}, K={K}):")
    print(f"  Volume in last {K}s: {avg_positions[-K:].sum():.1f} ({avg_positions[-K:].sum()/volume*100:.1f}%)")
    start = 60 - K
    top3 = np.argsort(avg_positions)[-3:][::-1]
    print(f"  Top 3 seconds: {', '.join(f't={s} ({avg_positions[s]/volume*100:.1f}%)' for s in top3)}")
    print(f"  Last second: {avg_positions[59]/volume*100:.1f}%")

: 

: 

: 

## Generate Test Positions

Run the agent on test data and save positions in the submission format.

In [ ]:
# Generate positions for all currencies
for currency in ALL_CURRENCIES:
    if currency not in all_results:
        continue

    val_insts = all_val[currency]
    volume = KNOWN_VOLUMES[currency]
    K = OPTIMAL_K[currency]
    Y_raw = all_Y_raw[currency]
    times = Y_raw["time_in_hour"].sort_values().unique()

    env = TradeExecutionEnv(val_insts)
    test_positions_df = pd.DataFrame()

    for i, inst in enumerate(val_insts):
        obs, _ = env.reset(options={"instrument_idx": i})
        positions = np.zeros(60)
        for step in range(K):
            action, _ = model.predict(obs, deterministic=True)
            pos = env.action_multipliers[int(action)] * env.twap_rate
            max_rem = volume - env.volume_allocated
            pos = min(pos, max(0.0, max_rem))
            if step == K - 1:
                pos = max(0.0, max_rem)
            abs_t = env.start_second + step
            positions[abs_t] = pos
            obs, _, done, _, _ = env.step(int(action))
            if done:
                break

        df = pd.DataFrame({
            'anonymized_id': inst['id'],
            'time_in_hour': times,
            'position': positions,
        })
        test_positions_df = pd.concat([test_positions_df, df], ignore_index=True)

    base_path = Path(f"positions/{currency}")
    base_path.mkdir(parents=True, exist_ok=True)
    target = base_path / "rl_cross_currency.parquet"
    test_positions_df.to_parquet(target, index=False, engine='pyarrow')
    print(f"  {currency}: {len(test_positions_df)} rows → {target}")

: 

: 

: 